In [1]:
import polars as pl
from hmmlearn import hmm
import matplotlib.pyplot as plt
import numpy as np
import plotly.express as px
import pandas as pd
import sys

In [2]:
from pathlib import Path

_root = Path.cwd()
while not (_root / "src").exists():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from src import config as src_config, data_utils, plots, strategy_utils

TRAIN_PATH = src_config.TRAIN_PATH
TEST_PATH = src_config.TEST_PATH

In [3]:
features = [
    "active_price",

    "realized_price",

    "mvrv",

    "mvrv_zscore",

    "market_cap",

    "supply_btc"
]

In [4]:
from arch import arch_model
from hmmlearn.hmm import GaussianHMM
from sklearn.preprocessing import StandardScaler
from pgmpy.models import BayesianNetwork, DiscreteBayesianNetwork
from pgmpy.estimators import MaximumLikelihoodEstimator
from pgmpy.inference import VariableElimination

C:\Users\Shruti\miniforge3\envs\adaptivesats\Lib\site-packages\pgmpy\estimators\__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in v1.3.0. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


In [5]:
from stacksats import BaseStrategy, StrategyRunner, BacktestConfig, ExportConfig, UniformStrategy, ComparisonConfig
runner = StrategyRunner()
uniform_strategy = UniformStrategy()

In [6]:
# If buying today was followed by relatively strong future appreciation,
# label today as an accumulation opportunity.

In [7]:
df = (
    pl.scan_parquet(TRAIN_PATH)
    .filter(pl.col("day_utc") >= pl.date(2018, 1, 1))
    .filter(pl.col("metric").is_in(features))
    .select(["day_utc", "metric", "value"])
    .collect()
    .pivot(
        on="metric",
        index="day_utc",
        values="value",
        aggregate_function="first",
    )
    .with_columns(
        (pl.col("market_cap") / pl.col("supply_btc")).alias("price_usd")
    )
    .rename({"day_utc": "date"})
    .filter(
        pl.col("price_usd").is_finite()
        & (pl.col("price_usd") > 0)
    )
    .sort("date")
)

df = df.to_pandas()

df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna(how="all")

df.isna().mean().sort_values(ascending=False)
df["date"] = pd.to_datetime(df["date"])

df = df.set_index("date").sort_index()
ts = df.copy()
ts["mvrv_deviation"] = ts["mvrv"] - ts["mvrv"].rolling(365).mean()
ts["mvrv_z_rolling"] = (
    (ts["mvrv"] - ts["mvrv"].rolling(365).mean())
    / ts["mvrv"].rolling(365).std()
)
model_df = ts.copy()

model_df["btc_log_return"] = np.log(
    model_df["price_usd"] / model_df["price_usd"].shift(1)
)

model_df["btc_return_pct"] = model_df["btc_log_return"] * 100

model_df = model_df.replace([np.inf, -np.inf], np.nan)
returns = model_df["btc_return_pct"].dropna()

In [8]:

garch = arch_model(
    returns,
    mean="Constant",
    vol="GARCH",
    p=1,
    q=1,
    dist="t"
)

garch_result = garch.fit(disp="off")

print(garch_result.summary())
model_df.loc[returns.index, "garch_volatility"] = garch_result.conditional_volatility
hmm_features = pd.DataFrame(index=model_df.index)

hmm_features["return"] = model_df["btc_log_return"]
hmm_features["volatility"] = model_df["garch_volatility"]
hmm_features["mvrv_z"] = model_df["mvrv_z_rolling"]
hmm_features["price_vs_active"] = (
    model_df["price_usd"] / model_df["active_price"] - 1
)

hmm_features = hmm_features.replace([np.inf, -np.inf], np.nan).dropna()
scaler = StandardScaler()

X = scaler.fit_transform(hmm_features)
hmm = GaussianHMM(
    n_components=3,
    covariance_type="full",
    n_iter=1000,
    random_state=42
)

hmm.fit(X)

regime_probs = hmm.predict_proba(X)
hmm_features["regime"] = hmm.predict(X)
hmm_features["price_vs_active"] = (
    model_df["price_usd"] / model_df["active_price"] - 1
)
for i in range(hmm.n_components):
    hmm_features[f"regime_prob_{i}"] = regime_probs[:, i]
prob_cols = [f"regime_prob_{i}" for i in range(hmm.n_components)]

plot_df = model_df.join(
    hmm_features[["regime","price_vs_active"] + prob_cols],
    how="left"
)

plot_df = plot_df.rename(columns={"regime": "hmm_regime"})

                        Constant Mean - GARCH Model Results                         
Dep. Variable:               btc_return_pct   R-squared:                       0.000
Mean Model:                   Constant Mean   Adj. R-squared:                  0.000
Vol Model:                            GARCH   Log-Likelihood:               -5576.05
Distribution:      Standardized Student's t   AIC:                           11162.1
Method:                  Maximum Likelihood   BIC:                           11190.6
                                              No. Observations:                 2190
Date:                      Sun, Jun 07 2026   Df Residuals:                     2189
Time:                              20:42:38   Df Model:                            1
                                 Mean Model                                
                 coef    std err          t      P>|t|     95.0% Conf. Int.
---------------------------------------------------------------------------
mu     

C:\Users\Shruti\miniforge3\envs\adaptivesats\Lib\site-packages\threadpoolctl.py:1226: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)


In [9]:
# We judge whether today was a good accumulation day based on what happens over the next 90 days.
# Using future data to CREATE labels is allowed.
# Using future data as MODEL INPUTS is leakage.

# we looked 90 days into the future to determine whether today was a good accumulation opportunity and the info
# was only used to assign the training label not as a model input. This is a valid approach because it simulates 
# the real-world scenario of making a buy decision today based on the expectation of future returns, 
# without giving the model access to future information at prediction time. 
# The model learns patterns in the historical data that are associated with good accumulation opportunities, 
# and then applies that knowledge to make predictions on new, unseen data.

1. Fixed-horizon labeling — Defining labels based on a fixed forward return window (your 90-day label_horizon) is the standard approach in supervised ML for financial time series. The alternative (variable-horizon labeling) is more complex and less common.
2. Quantile-based thresholding — Using the 65th percentile to define "accumulation opportunity" is a defensible, data-driven threshold. It avoids arbitrary cutoffs and ensures class balance (roughly 35% positive labels), which matters for classifier training.
3. Forward returns as labels — This is the canonical formulation. You're essentially asking: "Given today's features, did buying today yield above-median returns 90 days later?" This is a binary classification framing of a regression target, which is common.



Fixed-time horizon is a common method used in labeling financial data, usually applied on time bars. The rate of return relative to t0t_0
t0​ over time horizon hh
h is the standard formulation, as described by López de Prado in *Advances in Financial Machine Learning* (2018).

In supervised learning for financial datasets, it is important to define the correct labels. The Fixed-Horizon method looks at the price series as a sequence of fixed interval bars and assigns a label based on the difference of the open/close price for that specific bar. A threshold can also be introduced where the label is assigned only if the open/close difference exceeds that threshold

In [10]:
label_horizon = 90

bn_df = plot_df.copy()

bn_df["future_price"] = bn_df["price_usd"].shift(-label_horizon) # Look 90 days into the future to see what happens after buying today.

bn_df["forward_return"] = (
    bn_df["future_price"] / bn_df["price_usd"] - 1
) # Calculate the forward return over the next 90 days.

bn_df["future_sats_per_dollar"] = 1 / bn_df["price_usd"]

threshold = bn_df["forward_return"].quantile(0.65) # Set the threshold for an accumulation opportunity at the 65th percentile of forward returns. 
# This means we label the top 35% of days (in terms of future performance) as accumulation opportunities.

bn_df["accumulation_opportunity"] = np.where(
    bn_df["forward_return"] >= threshold,
    "yes",
    "no"
) # Label days as "yes" for accumulation opportunity if their forward return is above the threshold, otherwise "no".

In [11]:
bn_df["valuation_state"] = pd.cut(
    bn_df["mvrv_z_rolling"],
    bins=[-np.inf, -1.0, 1.0, np.inf],
    labels=["cheap", "neutral", "expensive"]
)

bn_df["volatility_state"] = pd.cut(
    bn_df["garch_volatility"].rank(pct=True),
    bins=[-np.inf, 0.33, 0.66, np.inf],
    labels=["low", "medium", "high"]
)

bn_df["active_price_state"] = pd.cut(
    bn_df["price_vs_active"],
    bins=[-np.inf, -0.05, 0.10, np.inf],
    labels=["below_active", "near_active", "above_active"]
)

bn_df["trend_state"] = np.where(
    bn_df["price_usd"] > bn_df["price_usd"].rolling(200).mean(),
    "uptrend",
    "downtrend"
)

In [12]:
for i in range(hmm.n_components):
    col = f"regime_prob_{i}"
    bn_df[f"{col}_state"] = pd.cut(
        bn_df[col],
        bins=[-np.inf, 0.33, 0.66, np.inf],
        labels=["low", "medium", "high"]
    )

In [13]:
prob_state_cols = [
    f"regime_prob_{i}_state"
    for i in range(hmm.n_components)
]

bn_cols = [
    "valuation_state",
    "volatility_state",
    "active_price_state",
    "trend_state",
    *prob_state_cols,
    "accumulation_opportunity",
]

bn_train = bn_df[bn_cols].dropna().astype(str)

In [14]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.inference import VariableElimination

edges = [
    ("valuation_state", "accumulation_opportunity"),
    ("volatility_state", "accumulation_opportunity"),
    ("active_price_state", "accumulation_opportunity"),
    ("trend_state", "accumulation_opportunity"),
]

for col in prob_state_cols:
    edges.append((col, "accumulation_opportunity"))

bayes_model = DiscreteBayesianNetwork(edges)

bayes_model.fit(bn_train)
bayes_model.check_model()

infer = VariableElimination(bayes_model)

In [15]:
def get_accumulation_probability(row):
    evidence = {
        col: str(row[col])
        for col in bn_cols
        if col != "accumulation_opportunity"
    }

    try:
        result = infer.query(
            variables=["accumulation_opportunity"],
            evidence=evidence,
            show_progress=False,
        )

        states = result.state_names["accumulation_opportunity"]
        values = result.values

        return float(dict(zip(states, values)).get("yes", 0.0))

    except Exception:
        return 0.0

In [16]:
bn_df["bayes_accumulation_prob"] = bn_df.apply(
    get_accumulation_probability,
    axis=1
)

In [17]:
# Squaring makes the strategy less aggressive at moderate confidence and much more aggressive only at high confidence.
# 1.8 controls overall aggressiveness.
# 1.0 is baseline DCA.

In [18]:
bn_df["accumulation_boost"] = (
    1.0 + 1.8 * (bn_df["bayes_accumulation_prob"] ** 2)
)

In [19]:
euphoric_regime = 0
accumulation_regime = 1
neutral_regime = 2

In [20]:
# Penalizes allocations during euphoric regimes
# High valuation, Strong profitability, Optimistic sentiment, Rapid price appreciation. During these periods BTC is often overvalued and prone to sharp corrections.

In [21]:
euphoric_prob_col = f"regime_prob_{euphoric_regime}"

bn_df["euphoria_penalty"] = (
    1 - 0.20 * bn_df[euphoric_prob_col]
).clip(0.80, 1.0)

In [22]:
bn_df["hmm_bayes_v2_allocation_weight"] = (
    bn_df["accumulation_boost"]
    * bn_df["euphoria_penalty"]
).clip(0.50, 2.50)

In [23]:
bn_df["hmm_bayes_v2_allocation_weight"] = (
    bn_df["hmm_bayes_v2_allocation_weight"]
    .rolling(7)
    .mean()
    .fillna(bn_df["hmm_bayes_v2_allocation_weight"])
)

In [24]:
signals_pd = (
    bn_df
    .reset_index()
    [["date", "hmm_bayes_v2_allocation_weight", "bayes_accumulation_prob"]]
)

btc_df_bayes_v2 = (
    df.reset_index()
    .merge(signals_pd, on="date", how="left")
)

btc_df_bayes_v2["hmm_bayes_v2_allocation_weight"] = (
    btc_df_bayes_v2["hmm_bayes_v2_allocation_weight"]
    .fillna(1.0)
)

In [25]:
btc_df_bayes_v2_pl = pl.from_pandas(btc_df_bayes_v2).with_columns(
    pl.col("date").cast(pl.Date)
)

In [26]:
class HMMBayesV2Strategy(BaseStrategy):
    strategy_id = "hmm-bayes-v2"
    version = "1.0.0"

    def __init__(self, signal_df, min_weight=0.50, max_weight=2.50):
        self.min_weight = min_weight
        self.max_weight = max_weight

        self.signal_df = (
            signal_df
            .select(["date", "hmm_bayes_v2_allocation_weight"])
            .with_columns(pl.col("date").cast(pl.Date))
        )

    def params(self):
        return {
            "min_weight": self.min_weight,
            "max_weight": self.max_weight,
            "signal_column": "hmm_bayes_v2_allocation_weight",
        }

    def propose_weight(self, state):
        current_date = (
            state.features
            .select("date")
            .tail(1)
            .item()
        )

        signal_row = self.signal_df.filter(
            pl.col("date") == current_date
        )

        if signal_row.is_empty():
            signal = 1.0
        else:
            signal = (
                signal_row
                .select("hmm_bayes_v2_allocation_weight")
                .item()
            )

        if signal is None:
            signal = 1.0

        signal = float(signal)
        signal = max(self.min_weight, min(signal, self.max_weight))

        return state.uniform_weight * signal

In [27]:
start_date = str(btc_df_bayes_v2_pl.select(pl.col("date").min()).item())
end_date = str(btc_df_bayes_v2_pl.select(pl.col("date").max()).item())

In [28]:
hmm_bayes_v2 = HMMBayesV2Strategy(signal_df=btc_df_bayes_v2_pl)

hmm_bayes_v2_bt = runner.backtest(
    hmm_bayes_v2,
    config=BacktestConfig(
        start_date=start_date,
        end_date=end_date,
    ),
    btc_df=btc_df_bayes_v2_pl,
)

In [29]:
hmm_bayes_v2_bt.to_dataframe()

window,min_sats_per_dollar,max_sats_per_dollar,uniform_sats_per_dollar,dynamic_sats_per_dollar,uniform_percentile,dynamic_percentile,excess_percentile
str,f64,f64,f64,f64,f64,f64,f64
"""2018-01-01 → 2018-12-31""",5842.827929,31422.726801,14736.064632,14736.064632,34.766505,34.766505,2.2027e-13
"""2018-01-02 → 2019-01-01""",5842.827929,31422.726801,14787.64918,14790.058625,34.968165,34.977584,0.009419
"""2018-01-03 → 2019-01-02""",5842.827929,31422.726801,14839.550357,14843.860001,35.171063,35.187911,0.016848
"""2018-01-04 → 2019-01-03""",5842.827929,31422.726801,14893.784195,14897.645116,35.383081,35.398174,0.015094
"""2018-01-05 → 2019-01-04""",5842.827929,31422.726801,14947.312798,14950.254323,35.592341,35.60384,0.011499
…,…,…,…,…,…,…,…
"""2022-12-28 → 2023-12-27""",2263.929335,6055.677106,3645.624389,3971.690862,36.43953,45.038901,8.599371
"""2022-12-29 → 2023-12-28""",2263.929335,6055.677106,3635.497492,3960.974754,36.172452,44.756284,8.583832
"""2022-12-30 → 2023-12-29""",2263.929335,6055.677106,3625.550662,3950.186492,35.910124,44.471765,8.561641


In [30]:
hmm_bayes_v2_bt.summary()

'Score: 61.47% | Win Rate: 77.07% | Exp-Decay Percentile: 45.87% | Uniform Exp-Decay: 37.29% | Exp-Decay Multiple: 1.230x | Windows: 1827'

In [31]:
strategies_train = {
    "dynamic": HMMBayesV2Strategy(signal_df=btc_df_bayes_v2_pl),
    "baseline": UniformStrategy(),
}

train_years  = range(src_config.TRAIN_START_YEAR, src_config.SPLIT_YEAR)   # 2018–2023
merged_train = strategy_utils.run_year_by_year(strategies_train, btc_df_bayes_v2_pl, train_years, runner)

perf_vf   = strategy_utils.compute_performance_summary(merged_train, "dynamic", "baseline")

print(f"[Train] HMM Bayes Look forward SPD : {perf_vf['sats_per_dollar_dynamic']:.2f}")
print(f"[Train] Baseline SPD    : {perf_vf['sats_per_dollar_baseline']:.2f}")
print(f"HMM Bayes Look forward {abs(perf_vf['pct_diff_vs_baseline']):.2f}% {perf_vf['performance_label']} than Uniform")

2018: exported 365 rows
2018: exported 365 rows


2019: exported 365 rows
2019: exported 365 rows


2020: exported 365 rows
2020: exported 365 rows


2021: exported 365 rows
2021: exported 365 rows


2022: exported 365 rows
2022: exported 365 rows


2023: exported 365 rows
2023: exported 365 rows
[Train] HMM Bayes Look forward SPD : 9119.03
[Train] Baseline SPD    : 8418.79
HMM Bayes Look forward 8.32% better than Uniform


## Adding Test data

In [32]:
# df_test = (
#     pl.scan_parquet(TEST_PATH)
#     .filter(pl.col("day_utc") >= pl.date(2024, 1, 1)) #Test data should anyways start from Jan 1, 2024
#     .filter(pl.col("metric").is_in(features))
#     .select(["day_utc", "metric", "value"])
#     .collect()
#     .pivot(
#         on="metric",
#         index="day_utc",
#         values="value",
#         aggregate_function="first",
#     )
#     .with_columns(
#         (pl.col("market_cap") / pl.col("supply_btc")).alias("price_usd")
#     )
#     .rename({"day_utc": "date"})
#     .filter(
#         pl.col("price_usd").is_finite()
#         & (pl.col("price_usd") > 0)
#     )
#     .sort("date")
# )


In [33]:
# strategies_test = {
#     "dynamic": HMMBayesV2Strategy(signal_df=btc_df_bayes_v2_pl),
#     "baseline": UniformStrategy(),
# }

# test_years  = range(src_config.SPLIT_YEAR, src_config.TEST_END_YEAR + 1)   # 2024–2025
# merged_test = strategy_utils.run_year_by_year(strategies_test, btc_df_bayes_v2_pl, test_years, runner)

# perf_test_vf   = strategy_utils.compute_performance_summary(merged_test, "dynamic", "baseline")

In [34]:
combined_plot_df = (
    pd.concat([merged_train.to_pandas(), 
               #merged_test.to_pandas()
               ])
    .sort_values("date")
    .reset_index(drop=True)
)
combined_plot_df["date"] = pd.to_datetime(combined_plot_df["date"])

cols = plots.StrategyColumns(
    weight="dynamic_weight",
    spd="sats_per_dollar_dynamic",
    sats_accum="sats_accum_dynamic",
)

# Full period
full_plot = plots.plot_strategy_full_period(
    combined_plot_df, 
    cols, 
    "HMM BAYES Look Forward Strategy", 
    date_range=("2018-01-01", str(combined_plot_df["date"].max().date())),
    #test_start_date="2024-01-01"
)
plt.show()